# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nazama-tech/Flyrank-Ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
import duckdb
import pandas as pd


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""CREATE SECRET (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )"""
)

print("Connected successfully!")

Connected successfully!


**Unit of analysis**

One row represents the performance of one content item for one client on one report date.

**Tables**

I will use fact_content_daily_performance for daily performance signals and dim_content for content-level attributes where needed.

**Time window**

I will use March 2026 as the development window and keep June 2026 as the sealed final month.

**Prediction**

I want to predict whether a piece of content will become a high-performing "hit", using information available before the prediction point.

**Deliberate exclusion**

I will exclude future outcome variables and target-derived fields because they would not be knowable at the decision moment and could cause leakage.

In [5]:
# This cell is for CODE (numbers, a query, a check).
df = con.sql("""
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    LIMIT 5
""").df()

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   report_date               5 non-null      datetime64[us]
 1   client_hash_id            5 non-null      object        
 2   content_hash_id           5 non-null      object        
 3   client_has_gsc            5 non-null      bool          
 4   client_has_ga4            5 non-null      bool          
 5   gsc_data_available        5 non-null      bool          
 6   ga4_data_available        5 non-null      bool          
 7   gsc_impressions           5 non-null      int64         
 8   gsc_clicks                5 non-null      int64         
 9   gsc_sum_position          5 non-null      int64         
 10  gsc_avg_position          5 non-null      float64       
 11  ga4_pageviews             5 non-null      int64         
 12  ga4_sessions              

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [6]:
# Grain check.
grain_check = con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


Grain check: The March 2026 data contains no duplicate combinations of report_date, client_hash_id, and content_hash_id. This supports the stated grain of one row per client, content item, and report date.

In [7]:
#How many rows are in my March slice
row_date_check = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

row_date_check

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


Row count and date-span check: The March 2026 slice contains 9,841,378 rows, with dates ranging from 2026-03-01 to 2026-03-31. This confirms that the selected slice covers the intended March 2026 window.

In [8]:
availability_columns = con.sql("""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
""").df()

availability_columns[
    availability_columns["column_name"].str.contains(
        "available", case=False, na=False
    )
]

,column_name,column_type,null,key,default,extra
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None


In [9]:
availability_check = con.sql("""
    SELECT
        COUNT(*) AS available_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


3,611,061 rows have gsc_data_available IS TRUE.



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
columns = con.sql("""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
""").df()

columns[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [11]:
columns[
    columns["column_name"].str.contains(
        "click|impression|position|view|session|conversion|trend|performance",
        case=False,
        na=False
    )
]

,column_name,column_type,null,key,default,extra
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None
10,gsc_avg_position,DOUBLE,YES,None,None,None
11,ga4_pageviews,BIGINT,YES,None,None,None
12,ga4_sessions,BIGINT,YES,None,None,None
14,ga4_engaged_sessions,BIGINT,YES,None,None,None
16,sessions_organic,BIGINT,YES,None,None,None
17,sessions_direct,BIGINT,YES,None,None,None
18,sessions_referral,BIGINT,YES,None,None,None


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.